# Diary Scraping & Word Extraction for Cursive Collection

This notebook provides tools to:
1. **Scrape diary pages** from americandiaryproject.com and save as PDF
2. **Extract word coordinates** using DeepSeek-OCR with provided transcripts

**For Google Colab**: All dependencies will be installed automatically.

## Part 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q selenium requests webdriver-manager Pillow pdf2image
!pip install -q torch torchvision transformers accelerate
!pip install -q einops easydict addict
!apt-get update -qq && apt-get install -y -qq poppler-utils
print("✅ Dependencies installed")

## Part 2: Diary Scraper

Downloads all 82 pages from the American Diary Project and saves as `diary.pdf`.

In [ ]:
import os
import time
import requests
import shutil
from PIL import Image
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Configuration
BASE_URL = "https://americandiaryproject.com/collection/1-10-2000-black-spiralbound-diary-from-a-new-yorker/"
TOTAL_PAGES = 82
TEMP_FOLDER = "diary_images"
OUTPUT_PDF = "diary.pdf"

def setup_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def download_image(url, filename):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            return True
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

def create_pdf(image_folder, output_pdf):
    print("Creating PDF...")
    image_files = sorted([f for f in os.listdir(image_folder) if f.endswith(".jpg")])
    
    if not image_files:
        print("No images found")
        return

    images = []
    for file in image_files:
        img = Image.open(os.path.join(image_folder, file))
        if img.mode != 'RGB':
            img = img.convert('RGB')
        images.append(img)

    if images:
        images[0].save(output_pdf, "PDF", resolution=100.0, save_all=True, append_images=images[1:])
        print(f"✅ PDF created: {output_pdf}")

def scrape_diary(pages=TOTAL_PAGES):
    if not os.path.exists(TEMP_FOLDER):
        os.makedirs(TEMP_FOLDER)

    driver = setup_driver()

    try:
        print(f"Scraping {pages} pages...")
        
        for page_num in range(1, pages + 1):
            target_url = f"{BASE_URL}?page_number_0={page_num}"
            driver.get(target_url)
            
            try:
                wait = WebDriverWait(driver, 10)
                img_element = wait.until(EC.presence_of_element_located(
                    (By.CLASS_NAME, "bwg_image_browser_img")
                ))

                img_url = img_element.get_attribute("src")
                
                if img_url:
                    file_name = f"{TEMP_FOLDER}/page_{page_num:02d}.jpg"
                    if download_image(img_url, file_name):
                        print(f"✅ Page {page_num}/{pages}")
                else:
                    print(f"⚠️ No image on page {page_num}")

            except Exception as e:
                print(f"⚠️ Error on page {page_num}: {e}")
            
            time.sleep(1)
        
        create_pdf(TEMP_FOLDER, OUTPUT_PDF)

    finally:
        driver.quit()
        if os.path.exists(TEMP_FOLDER):
            shutil.rmtree(TEMP_FOLDER)
        print("Complete!")

print("Scraper functions loaded. Call scrape_diary() to run.")

In [ ]:
# Run the scraper (this will take ~5-10 minutes for 82 pages)
# You can specify fewer pages for testing: scrape_diary(5)
scrape_diary()

## Part 3: Word Extraction with DeepSeek-OCR

Extract word bounding boxes using DeepSeek-OCR's grounding capabilities.

In [ ]:
import json
import torch
from pdf2image import convert_from_path
from transformers import AutoModelForCausalLM, AutoProcessor

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model (this will download ~3-6GB on first run)
print("Loading DeepSeek-OCR...")
processor = AutoProcessor.from_pretrained("deepseek-ai/DeepSeek-OCR", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-OCR", 
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print("✅ Model loaded")

def extract_words_from_page(image, transcript):
    """Extract bounding boxes for words in transcript."""
    words = transcript.split()
    results = []
    
    print(f"Processing {len(words)} words...")
    
    for i, word in enumerate(words):
        prompt = f"<image>\n<|grounding|>Locate <|ref|>{word}<|/ref|>"
        
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False
            )
            
        generated_text = processor.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        results.append({
            "word": word,
            "raw_output": generated_text
        })
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{len(words)} words")
        
    return results

def extract_from_pdf(pdf_path, transcripts, pages=None, output="words.json"):
    """Extract words from specified pages.
    
    Args:
        pdf_path: Path to PDF file
        transcripts: Dict mapping page numbers (as strings) to transcript text
        pages: List of page numbers to process (None = all pages in transcripts)
        output: Output JSON file path
    """
    if pages is None:
        pages = [int(p) for p in transcripts.keys()]
    
    all_results = {}
    
    for page_num in pages:
        str_page = str(page_num)
        if str_page not in transcripts:
            print(f"Skipping page {page_num}: No transcript")
            continue
            
        print(f"\nProcessing page {page_num}...")
        
        try:
            images = convert_from_path(pdf_path, first_page=page_num, last_page=page_num)
            if not images:
                print(f"Could not load page {page_num}")
                continue
                
            image = images[0]
            transcript = transcripts[str_page]
            
            page_results = extract_words_from_page(image, transcript)
            all_results[str_page] = page_results
            
        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            
    with open(output, 'w') as f:
        json.dump(all_results, f, indent=2)
        
    print(f"\n✅ Results saved to {output}")
    return all_results

print("Word extraction functions loaded.")

### Example: Extract words from pages

You need to provide transcripts for the pages you want to process.

In [ ]:
# Example transcripts (replace with actual transcripts)
transcripts = {
    "1": "Diary",
    "2": "January 10 2000",
    "3": "Today was a good day"
}

# Extract words from pages 1-3
results = extract_from_pdf(
    pdf_path="diary.pdf",
    transcripts=transcripts,
    pages=[1, 2, 3],
    output="word_coordinates.json"
)

# Display results
print(json.dumps(results, indent=2))

## Download Results

Download the generated files to your local machine.

In [ ]:
from google.colab import files

# Download PDF
if os.path.exists('diary.pdf'):
    files.download('diary.pdf')
    print("Downloaded diary.pdf")

# Download word coordinates
if os.path.exists('word_coordinates.json'):
    files.download('word_coordinates.json')
    print("Downloaded word_coordinates.json")